# Задание
## №1 Взвешивание дасететов (7 баллов)
Необходимо реализовать процедуру взвешивания датасетов соответствующую следующим критериям:
* В рамках одной эпохи (псевдоэпохи) доля киргизского языка должно быть в обучении > 30%
* В рамках одной эпохи (псевдоэпохи) доля киргизского языка должно быть в обучении < 50%
* В рамках одной эпохи (псевдоэпохи) доля fleurs_ky в киргизской части должна быть около половины (0.4 < fleurs_ky < 0.5)
* В рамках одной эпохи (псевдоэпохи) доля fleurs_ru в русской части должна быть в рамках 0.2 < fleurs_ru < 0.3

## №2 Управление нормой градиента (3 балла)
* Необходимо реализовать процедуру накопления градиента, так чтобы средняя норма градиента за эпоху не превышала значение 5)

**Эпохой** будем считать проход по всем данным. Т.е. в процессе обучения каждый пример был просмотрен хотя бы 1 раз (но может и больше).

**Псевдоэпохой** будем считать проход по количеству данных равному количеству примеров всех датасетов (len(fleurs_ky) + len(fleurs_ru) + len(common_voice_ru) + len(common_voice_ky)). В данном варианте проход по всем данным хотя бы 1 раз не гарантирован.

In [1]:
# !tar -xzf dls.tar.gz
# !pip install evaluate
# !pip install jiwer
!pip install resampy
# !pip install torchmetrics
# !pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 29.1 MB/s eta 0:00:0000:0100:01


In [2]:
# !pip uninstall transformers -y
# !pip install transformers==4.52.4
# !pip uninstall -y torch torchvision torchaudio
# !pip install torch==2.7.0 torchvision==0.22.0 torchaudio==2.7.0

In [2]:
from functools import cached_property
from pathlib import Path
from typing import Any, Dict, List, Optional, Union
import torch
import resampy
import numpy as np
import pandas as pd
import soundfile as sf
from transformers import (
    WhisperTokenizer,
    WhisperForConditionalGeneration,
    WhisperProcessor,
    BatchFeature
)
from torch.utils.data import WeightedRandomSampler


DATASET_ROOT = "/kaggle/input/datasets/foookushka/dls-ru-ky"

class MultiligualTokenizer:

    def __init__(self, tokenizer: WhisperTokenizer):
        self._tokenizer = tokenizer
        self._lang2code = {language: f"{code}" for language, code in TO_LANGUAGE_CODE.items()}
        self.vocab = self._tokenizer.get_vocab()

    def language_token(self, language: str) -> str:
        language = language.lower()
        if language not in self._lang2code:
            raise KeyError(f"Language {language} not found in tokenizer.")
        return f"<|{self._lang2code[language]}|>"

    def language_id(self, language: str) -> int:
        return self.vocab[self.language_token(language)]

    @cached_property
    def sot(self) -> int:
        return self.vocab["<|startoftranscript|>"]

    @cached_property
    def eot(self) -> int:
        return self.vocab["<|endoftext|>"]

    @cached_property
    def no_timestamps(self) -> int:
        return self.vocab["<|notimestamps|>"]

    @cached_property
    def transcribe(self) -> int:
        return self.vocab["<|transcribe|>"]

    def tokenize(self, text: str, language: str) -> Dict[str, List[int]]:
        text_tokens = self._tokenizer.encode(" " + text.strip(), add_special_tokens=False)
        sot_sequence = [self.sot, self.language_id(language), self.transcribe, self.no_timestamps]
        return sot_sequence + text_tokens + [self.eot]



class WhisperDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        manifests_files: List[str],
        languages: List[str],
        processor: WhisperProcessor,
        group_weights: Optional[List[float]] = None,
        dataset_name: Optional[str] = None,
        sampling_rate: Optional[int] = None
    ):
        assert len(manifests_files) == len(languages)
        if sampling_rate is None:
            sampling_rate = 16000
        if group_weights is None:
            group_weights = [1.0 / len(manifests_files)] * len(manifests_files)
        self.sampling_rate = sampling_rate
        self.dataset_name = dataset_name
        self.tokenizer = MultiligualTokenizer(tokenizer=processor.tokenizer)
        self.feature_extractor = processor.feature_extractor
        
        self.data = []
        group_sizes = []
        for i, (lang, path) in enumerate(zip(languages, manifests_files)):
            df = pd.read_csv(path, delimiter="\t")
            dataset_root = Path(DATASET_ROOT)
            group_sizes.append(len(df))
            for _, row in df.iterrows():
                audio_path = Path(row.path)
                # if manifest already contains relative dataset path
                if not audio_path.is_absolute():
                    audio_path = dataset_root / audio_path
                self.data.append({
                    "dataset_id": i,
                    "path": str(audio_path),
                    "transcription": row.transcription,
                    "lang": lang,
                    "dataset_name": dataset_name[i] if dataset_name is not None else None,
                })
        
        self.sample_weights = [
            group_weights[item["dataset_id"]] / group_sizes[item["dataset_id"]]
            for item in self.data
        ]

    def __getitem__(self, idx) -> Dict[str, Any]:
        item = self.data[idx]
        audio = self._read_audio(item["path"])
        return {
            "labels": self.tokenizer.tokenize(text=item["transcription"], language=item["lang"]),
            "input_features": self.feature_extractor(
                audio,
                sampling_rate=self.sampling_rate,
                padding="max_length"
            ).input_features[0],
            "language": item["lang"],
            "dataset_name": item["dataset_name"],
        }

    def _read_audio(self, audio_file):
        audio, sr = sf.read(audio_file)
        if len(audio.shape) == 2:
            audio = np.mean(audio, axis=1)
        if sr != self.sampling_rate:
            audio = resampy.resample(audio, sr, self.sampling_rate)
        return audio

    def __len__(self):
        return len(self.data)


class WhisperDataCollator:
    def __call__(
        self, inputs: List[Dict[str, Union[List[int], torch.Tensor]]]
    ) -> BatchFeature:
        # Extract input features and labels from the samples

        # Pad features
        input_features: List[np.ndarray] = [input["input_features"] for input in inputs]
        input_features_batch = np.stack(input_features, axis=0)
        input_features_batch = torch.FloatTensor(input_features_batch)

        # Pad labels
        labels: List[List[int]] = [input["labels"] for input in inputs]
        lengths = [len(label) for label in labels]
        max_length = max(lengths)
        labels_padded = [label + [-100] * (max_length - len(label)) for label in labels]
        labels_padded = torch.LongTensor(labels_padded)

        languages: List[str] = [input["language"] for input in inputs]
        dataset_names: List[str] = [input["dataset_name"] for input in inputs]

        attention_mask = torch.ones(
            input_features_batch.shape[:2],
            dtype=torch.long
        )
        
        return BatchFeature({
                "input_features": input_features_batch,
                "attention_mask": attention_mask,
                "labels": labels_padded,
                "language": languages,
                "dataset_name": dataset_names,
            })

In [3]:
from tqdm.notebook import tqdm
from collections import Counter
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchmetrics.text import WordErrorRate


class TrainingConfig:
    # Пути
    model_name = "openai/whisper-small"
    dataset_path = "./data"
    output_dir = "./whisper-finetuned"

    # Гиперпараметры
    batch_size = 8
    eval_batch_size = 4
    learning_rate = 1e-4
    num_epochs = 3
    warmup_steps = 500
    
    max_grad_norm = 4.9 #1.0
    gradient_accumulation_steps = 4
    
    weight_decay = 0.01

    # Настройки данных
    max_length = 448
    max_target_length = 128
    sampling_rate = 16000


class Trainer:

    def __init__(
            self,
            model: WhisperForConditionalGeneration,
            processor: WhisperProcessor,
            train_dataloader: DataLoader,
            test_dataloaders: Dict[str, DataLoader],
            config: TrainingConfig,
        ):
        self.grads = []
        self.langs = []
        self.datasets = []

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)
        self.processor = processor
        self.config = config
        self.optimizer = optim.AdamW(
            self.model.parameters(),
            lr=config.learning_rate,
            weight_decay=config.weight_decay,
        )
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer,
            T_max=len(train_dataloader) * self.config.num_epochs
        )
        self.train_dataloader = train_dataloader
        self.test_dataloaders = test_dataloaders
        self.wer_metric = WordErrorRate()
        self.scaler = torch.amp.GradScaler('cuda')


    def train_step(self):
        epoch_loss = 0.
        self.model.train()
    
        languages = []
        dataset_names = []
        grad_norms = []
    
        self.optimizer.zero_grad(set_to_none=True)
    
        for step, batch in enumerate(tqdm(self.train_dataloader)):
            input_features = batch["input_features"].to(self.device)
            labels = batch["labels"].to(self.device)
            attention_mask=batch["attention_mask"].to(self.device)
    
            languages += batch["language"]
            dataset_names += batch["dataset_name"]

            with torch.amp.autocast("cuda"):
                outputs = self.model(
                    input_features=input_features,
                    labels=labels,
                    attention_mask=attention_mask,
                    return_dict=True
                )
    
                loss = outputs.loss / self.config.gradient_accumulation_steps
    
            self.scaler.scale(loss).backward()
    
            should_step = (
                (step + 1) % self.config.gradient_accumulation_steps == 0
                or (step + 1) == len(self.train_dataloader)
            )
    
            if should_step:
                self.scaler.unscale_(self.optimizer)
            
                pre_clip_norm = torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(),
                    self.config.max_grad_norm,
                )
                
                post_clip_norm = min(float(pre_clip_norm), self.config.max_grad_norm)
                grad_norms.append(post_clip_norm)
            
                self.scaler.step(self.optimizer)
                self.scaler.update()
            
                self.scheduler.step()
                self.optimizer.zero_grad(set_to_none=True)

            epoch_loss += loss.item() * self.config.gradient_accumulation_steps

        assert sum(grad_norms) / len(grad_norms) < 5., "Норма градиента должна быть ниже 5"
        self.grads.append(grad_norms)
        cnt_lang = Counter(languages)
        cnt_dataset = Counter(dataset_names)
        self.langs.append(cnt_lang)
        self.datasets.append(cnt_dataset)

        ky_share = cnt_lang["kyrgyz"] / len(languages)
        assert ky_share > 0.3, "Киргизского языка должно быть в обучении > 30%"
        assert ky_share < 0.5 , "Киргизского языка должно быть в обучении < 50%"
        # assert (cnt_lang["fleurs_ky"] + cnt_dataset["common_voice_ky"]) / len(languages) > 0.3, "Киргизского языка должно быть в обучении > 30%"
        # assert (cnt_lang["fleurs_ky"] + cnt_dataset["common_voice_ky"]) / len(languages) < 0.5, "Киргизского языка должно быть в обучении < 50%"

        assert cnt_dataset["fleurs_ky"] / (cnt_dataset["common_voice_ky"] + cnt_dataset["fleurs_ky"]) > 0.4, "Доля fleurs_ky в киргизской части должна быть около половины"
        assert cnt_dataset["fleurs_ky"] / (cnt_dataset["common_voice_ky"] + cnt_dataset["fleurs_ky"]) < 0.5, "Доля fleurs_ky в киргизской части должна быть около половины"

        assert cnt_dataset["fleurs_ru"] / (cnt_dataset["common_voice_ru"] + cnt_dataset["fleurs_ru"]) > 0.2, "Доля fleurs_ru в русской должна быть > 0.2"
        assert cnt_dataset["fleurs_ru"] / (cnt_dataset["common_voice_ru"] + cnt_dataset["fleurs_ru"]) < 0.3, "Доля fleurs_ky в русской должна быть < 0.3"

        avg_loss = epoch_loss / len(self.train_dataloader)
        return avg_loss
        

    @torch.no_grad()
    def eval_step(self):
        self.model.eval()
        res = {}
        for name, test_dataloader in self.test_dataloaders.items():
            all_predictions = []
            all_references = []
            for batch in tqdm(test_dataloader):
                input_features = batch["input_features"].to(self.device)
                labels = batch["labels"]
                labels[labels == -100] = self.processor.tokenizer.eos_token_id #50257
                attention_mask = batch["attention_mask"].to(self.device)

                # Генерация
                generated_ids = self.model.generate(
                    input_features=input_features,
                    max_length=self.config.max_target_length,
                    language=batch["language"],
                    attention_mask=attention_mask,
                    num_beams=1
                )
                    # Декодирование
                predictions = self.processor.batch_decode(
                    generated_ids,
                    skip_special_tokens=True
                )
                references = self.processor.batch_decode(
                    labels,
                    skip_special_tokens=True
                )
                all_predictions.extend(predictions)
                all_references.extend(references)
            wer = self.wer_metric(all_predictions, all_references)
            res[name] = wer
        return res


    def train(self, epoch: int):
        train_losses = []
        eval_wers = []
        eval_wer = self.eval_step()
        eval_wers.append(eval_wer)
        for i in tqdm(range(epoch)):
            train_loss = self.train_step()
            train_losses.append(train_loss)
            eval_wer = self.eval_step()
            eval_wers.append(eval_wer)
        return eval_wers

In [5]:
from transformers.models.whisper.tokenization_whisper import TO_LANGUAGE_CODE, LANGUAGES


def update_vocab(model: WhisperForConditionalGeneration, processor: WhisperProcessor):
    cnt_new_tokens = 0
    for i, (code, language) in enumerate(NEW_LANGUAGES.items()):
        token = f"<|{code}|>"
        cnt = processor.tokenizer.add_tokens(token, special_tokens=True)
        if cnt == 1:
            cnt_new_tokens += cnt
            model.generation_config.lang_to_id[token] =  processor.tokenizer.get_vocab()[token]
    model.resize_token_embeddings(len(processor.tokenizer))
    return cnt_new_tokens


NEW_LANGUAGES = {"ky": "kyrgyz"}
NEW_TO_LANGUAGE_CODE = {"kyrgyz": "ky"}

LANGUAGES.update(NEW_LANGUAGES)
TO_LANGUAGE_CODE.update(NEW_TO_LANGUAGE_CODE)

MODEL_NAME = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(MODEL_NAME)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.gradient_checkpointing_enable()
model.config.use_cache = False

cnt_new_tokens = update_vocab(model, processor)

# ky = fleurs_ky + common_voice_ky = 0.18 + 0.22 = 0.40
# fleurs_ky / ky = 0.18 / 0.40 = 0.45
# ru = fleurs_ru + common_voice_ru = 0.15 + 0.45 = 0.60
# fleurs_ru / ru = 0.15 / 0.60 = 0.25
group_weights = [
    0.15,  # fleurs_ru
    0.18,  # fleurs_ky
    0.22,  # common_voice_ky
    0.45,  # common_voice_ru
]

dataset = WhisperDataset(
    manifests_files=[
        "/kaggle/input/datasets/foookushka/dls-ru-ky/dls/fleurs/ru/train/manifest.tsv",
        "/kaggle/input/datasets/foookushka/dls-ru-ky/dls/fleurs/ky/train/manifest.tsv",
        "/kaggle/input/datasets/foookushka/dls-ru-ky/dls/common_voice/ky/train/manifest.tsv",
        "/kaggle/input/datasets/foookushka/dls-ru-ky/dls/common_voice/ru/train/manifest.tsv",
    ],
    languages=[
        "russian",
        "kyrgyz",
        "kyrgyz",
        "russian",
    ],
    dataset_name=[
        "fleurs_ru",
        "fleurs_ky",
        "common_voice_ky",
        "common_voice_ru",
    ],
    group_weights=group_weights,
    processor=processor,
)

eval_datasets = {
    "fleurs_ru": WhisperDataset(["/kaggle/input/datasets/foookushka/dls-ru-ky/dls/fleurs/ru/test/manifest.tsv"], ["russian"], processor),
    "fleurs_ky": WhisperDataset(["/kaggle/input/datasets/foookushka/dls-ru-ky/dls/fleurs/ky/test/manifest.tsv"], ["kyrgyz"], processor),
    "common_voice_ru": WhisperDataset(["/kaggle/input/datasets/foookushka/dls-ru-ky/dls/common_voice/ru/test/manifest.tsv"], ["russian"], processor),
    "common_voice_ky": WhisperDataset(["/kaggle/input/datasets/foookushka/dls-ru-ky/dls/common_voice/ky/test/manifest.tsv"], ["kyrgyz"], processor),
}
config = TrainingConfig()

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(dataset.sample_weights),
    num_samples=len(dataset),   # pseudoepoch size
    replacement=True,
)

train_dataloader = DataLoader(
            dataset,
            batch_size=config.batch_size,
            sampler=sampler,
            shuffle=False,  # sampler и shuffle вместе не нужны
            collate_fn=WhisperDataCollator(),
            num_workers=0
)
test_dataloaders = {name: DataLoader(
            eval_dataset,
            batch_size=config.eval_batch_size,
            shuffle=True,
            collate_fn=WhisperDataCollator(),
            num_workers=0
) for (name, eval_dataset) in eval_datasets.items()}



trainer = Trainer(model, processor, train_dataloader, test_dataloaders, config)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Put 1 epoch, because it's too long trained \
(i trained on 3 epochs, but kaggle shutted down kernel and i lost all progress)

In [6]:
config.num_epochs = 1
res = trainer.train(config.num_epochs)

  0%|          | 0/135 [00:00<?, ?it/s]

A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


  0%|          | 0/176 [00:00<?, ?it/s]

  0%|          | 0/1746 [00:00<?, ?it/s]

  0%|          | 0/115 [00:00<?, ?it/s]

...

/tmp/ipykernel_57/3263661588.py:114: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  self.scheduler.step()


  0%|          | 0/135 [00:00<?, ?it/s]

  0%|          | 0/176 [00:00<?, ?it/s]

  0%|          | 0/1746 [00:00<?, ?it/s]

  0%|          | 0/115 [00:00<?, ?it/s]

In [7]:
res

[{'fleurs_ru': tensor(0.3206),
  'fleurs_ky': tensor(1.1410),
  'common_voice_ru': tensor(0.4027),
  'common_voice_ky': tensor(1.3177)},
 {'fleurs_ru': tensor(0.2363),
  'fleurs_ky': tensor(0.4648),
  'common_voice_ru': tensor(0.2238),
  'common_voice_ky': tensor(0.5200)}]

In [11]:
input_features = test_dataloaders["fleurs_ky"].dataset[212]["input_features"]
input_features = torch.from_numpy(input_features).unsqueeze(0).cuda()
generated_ids = model.generate(
                    input_features=input_features,
                    max_length=config.max_target_length,
                    language="ky",
                    task="transcribe",
                    num_beams=1
                )

In [12]:
generated_ids

tensor([[ 2344,   698,  1820,   386,  1069,   143,   102,  1354,   143,   102,
         49248,  1906,  1416,  2989,  1227,  9031,  2989,   143,   102,   143,
           102,  1268,  2019,  4219, 49248,  1906,  1416,  2989,  1227,  1070,
          5018,  1704,   693,  5740,  1094,  6251,   142,    96,  5572]],
       device='cuda:0')

In [13]:
batch = next(iter(test_dataloaders["fleurs_ky"]))

In [14]:
input_features = batch["input_features"].cuda()
labels = batch["labels"]
labels[labels == -100] = 50257

# Генерация
generated_ids = model.generate(
    input_features=input_features,
    max_length=config.max_target_length,
    language=batch["language"],
    num_beams=1)

In [15]:
processor.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

[' аткырылган иш теориялык мүнөздө болгонуна карабастан программа же ачыкалык тикасына карата түзүлгөн байкалордун маанилүүн түзүү максатында жазылган',
 ' олибер сакс президенттин сөзү деген макаласында мээк киргизген сыяктуу зыяндан улам кетти түшүнө албаган адамды ты буу б о карабы тан чынчылдыкты так бааларын көрсөткөн',
 ' пирамидалардын бетинде ар башка көрүнүш төрсөтөгүп ар башка пирамидаларга жыгы тигип турат',
 ' эртерээк келип кемпингтеги иш чарага жана музыкага жакын жайгашкан орунду э элесеңиз болот']

In [16]:
processor.tokenizer.batch_decode(batch["labels"], skip_special_tokens=True)

[' аткарылган иш теориялык мүнөздө болгонуна карабастан программа жаачы галактикасына карата жүргүзүлгөн байкоолордун моделин түзүү максатында жазылган',
 ' оливер сакс президенттин сөзү деген макаласында мээге келтирген зыяндан улам кепти түшүнө албаган адамдардын буга карабастан чынчылдыкты так баалай аларын көрсөткөн',
 ' пирамидалардын бетинде ар башка көрүнүштөр көрсөтүлүп ар башка пирамидаларга жарык тийип турат',
 ' эртерээк келип кемпингдеги иш чарага жана музыкага жакын жайгашкан орунду ээлесеңиз болот']

In [17]:
batch["language"]

['kyrgyz', 'kyrgyz', 'kyrgyz', 'kyrgyz']

In [21]:
import soundfile as sf

path = "/kaggle/input/datasets/foookushka/dls-ru-ky/dls/fleurs/ky/train/files/1.wav"
audio, sr = sf.read(path)

inputs = processor(
    audio,
    sampling_rate=sr,
    return_tensors="pt",
)

forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="kyrgyz",
    task="transcribe"
)

generated_ids = model.generate(
    inputs.input_features.to(model.device),
    forced_decoder_ids=forced_decoder_ids,
)

transcription = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True
)

print(transcription)

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.


[' добуш октябрь аында копенгагенде өтө турган отурумда эок тарабынан толугу менен ратификацияланыш керек']


In [22]:
inputs

{'input_features': tensor([[[-1.5000, -1.5000, -1.5000,  ..., -1.5000, -1.5000, -1.5000],
         [-1.5000, -1.5000, -1.5000,  ..., -1.5000, -1.5000, -1.5000],
         [-1.5000, -1.5000, -1.5000,  ..., -1.5000, -1.5000, -1.5000],
         ...,
         [-1.5000, -1.5000, -1.5000,  ..., -1.5000, -1.5000, -1.5000],
         [-1.5000, -1.5000, -1.5000,  ..., -1.5000, -1.5000, -1.5000],
         [-1.5000, -1.5000, -1.5000,  ..., -1.5000, -1.5000, -1.5000]]])}